# Signal Processing

In this notebook I will analyze two main topics in signal processing: filters, and the envelope. In addition, the power spectrum will be used to see how the previous signal processing algorithms change a signal also in the frequency-domain. These are the algorithms that will be analyzed:
- Low pass filter
- High pass filter
- Band pass filter
- Envelope

## Environment set up

Change the working directory to be able to work with the source-code of this repository.

In [1]:
import os
from pathlib import Path

WORKING_DIRECTORY = Path.cwd().parents[0]
os.chdir(WORKING_DIRECTORY)

## Imports

In [2]:
from src.read import read_nasa_vibration_files_in_directory
from src.signals.processing import Signal, band_pass_filter, low_pass_filter, high_pass_filter, envelope, power_spectrum, process_signal
from typing import Optional
from src.signals import calculations
import matplotlib.pyplot as plt
import numpy as np
from loguru import logger
import matplotlib.dates as mdates
import polars as pl
import plotly.express as px
import pandas as pd
from plotly.subplots import make_subplots
from datetime import datetime

## Inputs

The inputs have been obtained from the NASA bearings documentation.

The following cell displays the data path for each test and the name of their columns:

In [3]:
DATA_INPUTS_PER_TEST = {
    '1st_test': {'data_path': 'data/nasa_ims_bearing_dataset/1st_test',
                  'column_names': ['channel_1', 'channel_2', 'channel_3', 'channel_4',
                                   'channel_5', 'channel_6', 'channel_7', 'channel_8']},
    '2nd_test': {'data_path': 'data/nasa_ims_bearing_dataset/2nd_test',
                 'column_names': ['channel_1', 'channel_2', 'channel_3', 'channel_4']},
    '3rd_test': {'data_path': 'data/nasa_ims_bearing_dataset/3rd_test/4th_test/txt',
                 'column_names': ['channel_1', 'channel_2', 'channel_3', 'channel_4']}
          }

As each test has a different set up of sensors or channels per bearing, the following cell describes them:

In [4]:
BEARING_CHANNEL_MAPPING = {
    '1st_test': {'bearing_1': ['channel_1', 'channel_2'],
                 'bearing_2': ['channel_3', 'channel_4'],
                 'bearing_3': ['channel_5', 'channel_6'],
                 'bearing_4': ['channel_7', 'channel_8']},
    '2nd_test': {'bearing_1': ['channel_1'],
                 'bearing_2': ['channel_2'],
                 'bearing_3': ['channel_3'],
                 'bearing_4': ['channel_4']},
    '3rd_test': {'bearing_1': ['channel_1'],
                 'bearing_2': ['channel_2'],
                 'bearing_3': ['channel_3'],
                 'bearing_4': ['channel_4']}                 
}

Next, the faulty bearings are defined per test:

In [5]:
FAULTY_BEARINGS_PER_TEST = {
    '1st_test': {'bearing_3': 'bearing_inner_race',
                 'bearing_4': 'bearing_roller'
                 },
    '2nd_test': {'bearing_1': 'bearing_outer_race'},
    '3rd_test': {'bearing_3': 'bearing_outer_race'}
}

As final inputs, the following parameters are needed to read properly the vibration signals. In addition, an acceptable sensor range is defined to avoid faulty channel signals:

In [6]:
SAMPLING_FREQUENCY = 20000
MEASUREMENT_DURATION_IN_SECONDS = 1
ACCEPTABLE_SENSOR_RANGE = 0.01

## Read the Data

In [7]:
complete_data_path_per_test = {}

for test, inputs_per_test in DATA_INPUTS_PER_TEST.items():
    for key, values in inputs_per_test.items():
        data_path = inputs_per_test['data_path']
        complete_path = WORKING_DIRECTORY.joinpath(data_path)
        complete_data_path_per_test[test] = complete_path


In [8]:
signal_resolution = calculations.resolution(sampling_frequency=SAMPLING_FREQUENCY)

df_list_per_test = {}
for test, file_path in complete_data_path_per_test.items():
    logger.info(f'test: {test}')
    column_names = DATA_INPUTS_PER_TEST[test]['column_names']
    df_list = read_nasa_vibration_files_in_directory(files_path=file_path, sensors=column_names,
                                                     signal_resolution=signal_resolution,
                                                     acceptable_sensor_range=ACCEPTABLE_SENSOR_RANGE)
    df_list_sorted = sorted(df_list, key=lambda df: datetime.strptime(df['file_name'][0], '%Y.%m.%d.%H.%M.%S')) 
    df_list_per_test[test] = df_list_sorted

2026-02-11 13:14:17.566 | INFO     | __main__:<module>:5 - test: 1st_test
2026-02-11 13:14:19.721 | INFO     | src.read:read_nasa_vibration_files_in_directory:142 - 0 files were discarded.
2026-02-11 13:14:19.722 | INFO     | src.read:read_nasa_vibration_files_in_directory:143 - 1092 files were read successfully.
2026-02-11 13:14:19.816 | INFO     | __main__:<module>:5 - test: 2nd_test
2026-02-11 13:14:20.416 | WARNING  | src.read:read_nasa_vibration_files_in_directory:128 - All sensors in file 2004.02.19.06.22.39 are faulty for the defined acceptable_sensor_range of 0.01. Skipping this file.
2026-02-11 13:14:20.790 | WARNING  | src.read:read_nasa_vibration_files_in_directory:128 - All sensors in file 2004.02.19.06.12.39 are faulty for the defined acceptable_sensor_range of 0.01. Skipping this file.
2026-02-11 13:14:20.880 | INFO     | src.read:read_nasa_vibration_files_in_directory:142 - 2 files were discarded.
2026-02-11 13:14:20.880 | INFO     | src.read:read_nasa_vibration_files_in

## Sample Signal

For the analysis, one sample signal will be taken.

In [9]:
TEST= '2nd_test'
CHANNEL = 'channel_1'

sample_signal_df= df_list_per_test[TEST][-1]  # Last measurement
x = sample_signal_df['measurement_time_in_seconds'].to_numpy()
y = sample_signal_df[CHANNEL].to_numpy()

sample_signal = Signal(x=x, y=y)
print(f'sample_signal: {sample_signal}')


sample_signal: Signal(x=array([0.00000e+00, 5.00000e-05, 1.00000e-04, ..., 1.02385e+00,
       1.02390e+00, 1.02395e+00], shape=(20480,)), y=array([-0.005, -0.347, -0.168, ..., -0.967, -0.876,  0.076],
      shape=(20480,)))


In [10]:
measurement_recording_date_time = sample_signal_df['file_name'][0]

measurement_duration = sample_signal.x[-1]
signal_resolution = calculations.resolution(sampling_frequency=SAMPLING_FREQUENCY)
signal_resolution_manual_calculation = sample_signal.x[1] - sample_signal.x[0]


print(f'measurement_duration: {measurement_duration} s')
print(f'signal_resolution: {signal_resolution} s')
print(f'signal_resolution_manual_calculation: {signal_resolution_manual_calculation} s')


measurement_duration: 1.0239500000000001 s
signal_resolution: 5e-05 s
signal_resolution_manual_calculation: 5e-05 s


In [11]:
df_sample = pd.DataFrame({
    'time': sample_signal.x,
    'amplitude': sample_signal.y
})

fig = px.line(df_sample, x='time', y='amplitude',
              title=f'Sample Raw Signal: {TEST} - {CHANNEL} - {measurement_recording_date_time}',
              labels={'time': 'Time (s)', 'amplitude': 'Amplitude (g)'})

fig.update_layout(
    height=500,
    hovermode='x unified'
)
fig.update_xaxes(title_text='Time (s)')
fig.update_yaxes(title_text='Amplitude (g)')
fig.show();

## Plotting Function

The following function has been created to facilitate the analysis:

In [12]:
def plot_signals_and_spectra(
    sample_signal: Signal,
    filtered_signal: Signal,
    sample_power_spectrum_signal: Signal,
    filtered_power_spectrum_signal: Signal,
    test: str,
    channel: str,
    measurement_recording_date_time: str,
    sampling_frequency: float,
    measurement_duration: float,
    low_cutoff_frequency: Optional[float] = None,
    high_cutoff_frequency: Optional[float] = None,
    filter_label: str = 'filtered signal',
    time_xtick_step: Optional[float] = 0.05,
    freq_xtick_step: Optional[float] = 500,
) -> None:
    """Create time-domain (overlaid) and frequency-domain (stacked, shared x-axis) plots using Plotly.

    Determines plot titles from which cutoff frequencies are provided:
    - low_cutoff_frequency only -> Low-pass
    - high_cutoff_frequency only -> High-pass
    - both provided -> Band-pass

    Parameters
    - sample_signal, filtered_signal: `Signal` objects with `.x` and `.y` (numpy arrays).
    - sample_power_spectrum_signal, filtered_power_spectrum_signal: `Signal` objects with `.x` and `.y`.
    - test, channel, measurement_recording_date_time: strings used in titles.
    - sampling_frequency, measurement_duration, low_cutoff_frequency, high_cutoff_frequency: numeric metadata.
    - filter_label: legend label for the filtered trace.
    - time_xtick_step, freq_xtick_step: tick spacing for x-axes.
    """
    from plotly.subplots import make_subplots

    # Decide filter type and build subtitle pieces
    if low_cutoff_frequency is not None and high_cutoff_frequency is not None:
        filter_type = 'Band-pass'
        cutoff_info = f'Low Cutoff Frequency: {low_cutoff_frequency} Hz - High Cutoff Frequency: {high_cutoff_frequency} Hz'
    elif low_cutoff_frequency is not None:
        filter_type = 'Low-pass'
        cutoff_info = f'Cutoff Frequency: {low_cutoff_frequency} Hz'
    elif high_cutoff_frequency is not None:
        filter_type = 'High-pass'
        cutoff_info = f'Cutoff Frequency: {high_cutoff_frequency} Hz'
    else:
        filter_type = 'Filtered'
        cutoff_info = ''

    # Create DataFrames for time domain signals
    df_raw = pd.DataFrame({'time': sample_signal.x, 'amplitude': sample_signal.y})
    df_filtered = pd.DataFrame({'time': filtered_signal.x, 'amplitude': filtered_signal.y})
    df_raw_spectrum = pd.DataFrame({'frequency': sample_power_spectrum_signal.x, 'amplitude': sample_power_spectrum_signal.y})
    df_filtered_spectrum = pd.DataFrame({'frequency': filtered_power_spectrum_signal.x, 'amplitude': filtered_power_spectrum_signal.y})

    # Time-domain plot
    fig_time = px.line()
    fig_time.add_scatter(x=df_raw['time'], y=df_raw['amplitude'], mode='lines', name='raw signal')
    fig_time.add_scatter(x=df_filtered['time'], y=df_filtered['amplitude'], mode='lines', name=filter_label)


    fig.update_xaxes(
        showspikes=True,
        spikemode="across",   # Draws the line across the entire plot area
        spikesnap="cursor",   # Snaps the line to the mouse cursor
        spikethickness=1
        )
    
    fig_time.update_layout(
        title=f'Sample Raw Signal: {test} - {channel} - {measurement_recording_date_time}',
        xaxis_title='Time (s)',
        yaxis_title='Amplitude (g)',
        height=500,
        hovermode='x unified',
        annotations=[dict(
            text=f'Sampling frequency: {sampling_frequency} Hz - Measurement duration: {measurement_duration:.4f} s'
                 + (f' - {cutoff_info}' if cutoff_info else ''),
            showarrow=False,
            xref='paper', yref='paper',
            x=0.5, y=-0.15, xanchor='center', yanchor='top'
        )]
    )
    
    fig_time.show();

    # Frequency-domain plot (stacked subplots)
    fig_freq = make_subplots(rows=2, cols=1, shared_xaxes=True,
                             subplot_titles=('Raw Signal Spectrum', f'{filter_label.capitalize()} Spectrum'),
                             vertical_spacing=0.15)
    
    fig_freq.add_scatter(x=df_raw_spectrum['frequency'], y=df_raw_spectrum['amplitude'], 
                         mode='lines+markers', name='raw signal', row=1, col=1,
                         line={'dash': 'dot'}, marker={'symbol': 'circle', 'size': 6, 'opacity': 0.8})
    fig_freq.add_scatter(x=df_filtered_spectrum['frequency'], y=df_filtered_spectrum['amplitude'], 
                         mode='lines+markers', name=filter_label, row=2, col=1,
                         line={'dash': 'dot'}, marker={'symbol': 'circle', 'size': 6, 'opacity': 0.8})
    
    fig_freq.update_yaxes(title_text='Amplitude (g)', row=1, col=1)
    fig_freq.update_yaxes(title_text='Amplitude (g)', row=2, col=1)
    fig_freq.update_xaxes(title_text='Frequency (Hz)', row=2, col=1)
    
    fig_freq.update_layout(
        title=f'Power Spectrum: {test} - {channel} - {measurement_recording_date_time}<br>'
              f'Sampling frequency: {sampling_frequency} Hz - Measurement duration: {measurement_duration:.4f} s'
              + (f' - {cutoff_info}' if cutoff_info else ''),
        height=600,
        hovermode='x unified',
        hoversubplots='axis'
    )
 

    fig_freq.show();

## Signal Processing

### Low-Pass Filter

A low-pass filter is a filter that passes signals with a frequency lower than a selected cutoff frequency and attenuates signals with frequencies higher than the cutoff frequency. The exact frequency response of the filter depends on the filter design. (Source: Wikipedia)

In [13]:
LOW_PASS_CUTOFF_FREQUENCY = 2500

low_pass_filtered_signal = low_pass_filter(signal=sample_signal, sampling_frequency=SAMPLING_FREQUENCY, cutoff_frequency=LOW_PASS_CUTOFF_FREQUENCY)

sample_power_spectrum_signal = power_spectrum(signal=sample_signal, sampling_frequency=SAMPLING_FREQUENCY)
low_pass_filtered_power_spectrum_signal = power_spectrum(signal=low_pass_filtered_signal, sampling_frequency=SAMPLING_FREQUENCY)


plot_signals_and_spectra(sample_signal, low_pass_filtered_signal, sample_power_spectrum_signal, low_pass_filtered_power_spectrum_signal, 
                      TEST, CHANNEL, measurement_recording_date_time, SAMPLING_FREQUENCY, measurement_duration, 
                      low_cutoff_frequency=LOW_PASS_CUTOFF_FREQUENCY, filter_label='low pass filtered signal')

The designed filter low pass filter removes the frequencies above the cut off frequency of 2500 Hz. Therefore, the spectrum in the values above the cut off frequency are removed and displayed as 0.

### High-Pass Filter

A high-pass filter (HPF) is an electronic filter that passes signals with a frequency higher than a certain cutoff frequency and attenuates signals with frequencies lower than the cutoff frequency. (Source: Wikipedia)

In [14]:
HIGH_PASS_CUTOFF_FREQUENCY = 2500

high_pass_filtered_signal = high_pass_filter(signal=sample_signal, sampling_frequency=SAMPLING_FREQUENCY, cutoff_frequency=HIGH_PASS_CUTOFF_FREQUENCY)

sample_power_spectrum_signal = power_spectrum(signal=sample_signal, sampling_frequency=SAMPLING_FREQUENCY)
high_pass_filtered_power_spectrum_signal = power_spectrum(signal=high_pass_filtered_signal, sampling_frequency=SAMPLING_FREQUENCY)


plot_signals_and_spectra(sample_signal, high_pass_filtered_signal, sample_power_spectrum_signal, high_pass_filtered_power_spectrum_signal,
                      TEST, CHANNEL, measurement_recording_date_time, SAMPLING_FREQUENCY, measurement_duration,
                      high_cutoff_frequency=HIGH_PASS_CUTOFF_FREQUENCY, filter_label='high pass filtered signal')

The designed filter high pass filter removes the frequencies lower the cut off frequency of 2500 Hz. Therefore, the spectrum in the values lower the cut off frequency are removed and displayed as 0.

### Band-Pass-Filter

A band-pass filter or bandpass filter (BPF) is a device that passes frequencies within a certain range and rejects (attenuates) frequencies outside that range. (Source: Wikipedia)

In [15]:
LOW_CUTOFF_FREQUENCY = 2500
HIGH_CUTOFF_FREQUENCY = 5000

high_pass_filtered_signal = band_pass_filter(signal=sample_signal, 
                                             sampling_frequency=SAMPLING_FREQUENCY, 
                                             low_cutoff_frequency=LOW_CUTOFF_FREQUENCY,
                                             high_cutoff_frequency=HIGH_CUTOFF_FREQUENCY)

sample_power_spectrum_signal = power_spectrum(signal=sample_signal, 
                                              sampling_frequency=SAMPLING_FREQUENCY)
band_pass_filtered_power_spectrum_signal = power_spectrum(signal=high_pass_filtered_signal, 
                                                          sampling_frequency=SAMPLING_FREQUENCY)

plot_signals_and_spectra(sample_signal, high_pass_filtered_signal, sample_power_spectrum_signal, band_pass_filtered_power_spectrum_signal,
                      TEST, CHANNEL, measurement_recording_date_time, SAMPLING_FREQUENCY, measurement_duration,
                      low_cutoff_frequency=LOW_CUTOFF_FREQUENCY, high_cutoff_frequency=HIGH_CUTOFF_FREQUENCY, 
                      filter_label='band pass filtered signal')

The band pass filter keeps the signal between the defined low and high cut off frequencies (2500, and 5000 Hz).

### Envelope

In [16]:
sample_envelope_signal = envelope(signal=sample_signal)
sample_envelope_with_dc_offset_signal = envelope(signal=sample_signal, remove_dc_offset=True)

sample_power_spectrum_signal = power_spectrum(signal=sample_signal, 
                                              sampling_frequency=SAMPLING_FREQUENCY)
envelope_power_spectrum_signal = power_spectrum(signal=sample_envelope_signal, 
                                                sampling_frequency=SAMPLING_FREQUENCY)
envelope_with_dc_offset_power_spectrum_signal = power_spectrum(signal=sample_envelope_with_dc_offset_signal, 
                                                sampling_frequency=SAMPLING_FREQUENCY)


In [17]:
import pandas as pd
from plotly.subplots import make_subplots

# Create DataFrames for plotly
df_raw = pd.DataFrame({
    'time': sample_signal.x,
    'amplitude': sample_signal.y
})

df_envelope = pd.DataFrame({
    'time': sample_envelope_signal.x,
    'amplitude': sample_envelope_signal.y
})

df_envelope_dc = pd.DataFrame({
    'time': sample_envelope_with_dc_offset_signal.x,
    'amplitude': sample_envelope_with_dc_offset_signal.y
})

# Create subplots with shared x-axis
fig = make_subplots(rows=3, cols=1, shared_xaxes=True,
                    subplot_titles=('Raw Signal', 'Envelope Signal', 'Envelope without DC Offset Signal'),
                    vertical_spacing=0.1)

# Add raw signal trace
fig.add_scatter(x=df_raw['time'], y=df_raw['amplitude'], 
                mode='lines', name='Raw Signal', 
                row=1, col=1)

# Add envelope signal trace
fig.add_scatter(x=df_envelope['time'], y=df_envelope['amplitude'], 
                mode='lines', name='Envelope Signal',
                row=2, col=1)

# Add envelope without DC offset trace
fig.add_scatter(x=df_envelope_dc['time'], y=df_envelope_dc['amplitude'], 
                mode='lines', name='Envelope without DC Offset', 
                row=3, col=1)

# Update y-axes labels
fig.update_yaxes(title_text='Amplitude (g)', row=1, col=1)
fig.update_yaxes(title_text='Amplitude (g)', row=2, col=1)
fig.update_yaxes(title_text='Amplitude (g)', row=3, col=1)
fig.update_xaxes(title_text='Time (s)', row=3, col=1)

fig.update_layout(height=900, width=1200, title_text='Signal Analysis: Raw vs Envelope Signals', hovermode='x unified', hoversubplots='axis')
fig.show();

In [18]:
plot_signals_and_spectra(sample_signal, sample_envelope_signal, sample_power_spectrum_signal, envelope_power_spectrum_signal,
                      TEST, CHANNEL, measurement_recording_date_time, SAMPLING_FREQUENCY, measurement_duration,
                      filter_label='envelope signal')

In [19]:
plot_signals_and_spectra(sample_signal, sample_envelope_with_dc_offset_signal, sample_power_spectrum_signal, 
                         envelope_with_dc_offset_power_spectrum_signal,
                      TEST, CHANNEL, measurement_recording_date_time, SAMPLING_FREQUENCY, measurement_duration,
                      filter_label='envelope with dc offset signal')